# Hofstede Individualism Scores

Downloads the official Hofstede 6-dimension dataset from geerthofstede.com and saves a
cleaned ISO3-keyed CSV to `data/gravity/hofstede.csv`. The file is used downstream to
construct team-level individualism scores (`IC_team = (IC_p1 + IC_p2) / 2`) for the
heterogeneity regressions.

In [ ]:
import io
import os
import requests
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
OUT_PATH = os.path.join(ROOT, 'data', 'gravity', 'hofstede.csv')
print('Output:', OUT_PATH)

## 1. Download raw Hofstede data

In [ ]:
URL = 'https://geerthofstede.com/wp-content/uploads/2016/08/6-dimensions-for-website-2015-08-16.csv'
headers = {'User-Agent': 'Mozilla/5.0 (compatible; research-bot/1.0)'}

resp = requests.get(URL, headers=headers, timeout=30)
resp.raise_for_status()

raw = pd.read_csv(io.StringIO(resp.text), sep=';')
raw.columns = raw.columns.str.strip().str.lower()
print(f'Raw rows: {len(raw)}, columns: {raw.columns.tolist()}')
raw.head()

## 2. Map Hofstede country codes → ISO 3166-1 alpha-3

Hofstede uses its own 3-letter codes (`ctr`) that differ from ISO3 for many countries.
Sub-national entries (e.g. Belgium French, Switzerland German) and regional aggregates
(Africa East, Arab countries) are excluded — we keep only the national aggregate.

In [ ]:
# Mapping: Hofstede ctr -> ISO3
# Sub-national / regional entries are omitted intentionally.
CTR_TO_ISO3 = {
    'ARG': 'ARG',  # Argentina
    'AUL': 'AUS',  # Australia
    'AUT': 'AUT',  # Austria
    'BAN': 'BGD',  # Bangladesh
    'BEL': 'BEL',  # Belgium (national aggregate; BEF/BEN are sub-national)
    'BRA': 'BRA',  # Brazil
    'BUL': 'BGR',  # Bulgaria
    'CAN': 'CAN',  # Canada (national aggregate; CAF is sub-national)
    'CHL': 'CHL',  # Chile
    'CHI': 'CHN',  # China
    'COL': 'COL',  # Colombia
    'COS': 'CRI',  # Costa Rica
    'CRO': 'HRV',  # Croatia
    'CZE': 'CZE',  # Czech Republic
    'DEN': 'DNK',  # Denmark
    'ECA': 'ECU',  # Ecuador
    'SAL': 'SLV',  # El Salvador
    'EST': 'EST',  # Estonia
    'FIN': 'FIN',  # Finland
    'FRA': 'FRA',  # France
    'GER': 'DEU',  # Germany
    'GBR': 'GBR',  # Great Britain
    'GRE': 'GRC',  # Greece
    'GUA': 'GTM',  # Guatemala
    'HOK': 'HKG',  # Hong Kong
    'HUN': 'HUN',  # Hungary
    'IND': 'IND',  # India
    'IDO': 'IDN',  # Indonesia
    'IRA': 'IRN',  # Iran
    'IRE': 'IRL',  # Ireland
    'ISR': 'ISR',  # Israel
    'ITA': 'ITA',  # Italy
    'JAM': 'JAM',  # Jamaica
    'JPN': 'JPN',  # Japan
    'KOR': 'KOR',  # South Korea
    'LAT': 'LVA',  # Latvia
    'LIT': 'LTU',  # Lithuania
    'LUX': 'LUX',  # Luxembourg
    'MAL': 'MYS',  # Malaysia
    'MLT': 'MLT',  # Malta
    'MEX': 'MEX',  # Mexico
    'MOR': 'MAR',  # Morocco
    'NET': 'NLD',  # Netherlands
    'NZL': 'NZL',  # New Zealand
    'NOR': 'NOR',  # Norway
    'PAK': 'PAK',  # Pakistan
    'PAN': 'PAN',  # Panama
    'PER': 'PER',  # Peru
    'PHI': 'PHL',  # Philippines
    'POL': 'POL',  # Poland
    'POR': 'PRT',  # Portugal
    'ROM': 'ROU',  # Romania
    'RUS': 'RUS',  # Russia
    'SER': 'SRB',  # Serbia
    'SIN': 'SGP',  # Singapore
    'SLK': 'SVK',  # Slovakia
    'SLV': 'SVN',  # Slovenia
    'SPA': 'ESP',  # Spain
    'SUR': 'SUR',  # Suriname
    'SWE': 'SWE',  # Sweden
    'SWI': 'CHE',  # Switzerland (national aggregate; SWF/SWG are sub-national)
    'TAI': 'TWN',  # Taiwan
    'THA': 'THA',  # Thailand
    'TRI': 'TTO',  # Trinidad and Tobago
    'TUR': 'TUR',  # Turkey
    'USA': 'USA',  # United States
    'URU': 'URY',  # Uruguay
    'VEN': 'VEN',  # Venezuela
    'VIE': 'VNM',  # Vietnam
    # South Africa: SAW is the white sub-group but is the only ZAF entry available
    'SAW': 'ZAF',
}

# ---------------------------------------------------------------------------
# Proxy entries for countries in the tennis dataset not covered by Hofstede.
#
# Proxy selection rationale (two tiers):
#
#   Tier 1 — direct cultural/linguistic kin (high confidence):
#     BIH (Bosnia & Herzegovina) -> SRB: South Slavic language family,
#       shared Yugoslav history, structurally identical institutional setting.
#       Players: Brkic, Dzumhur, Basic (36 player-slots in dataset).
#     CYP (Cyprus) -> GRC: Greek-Cypriot majority (~80% of population),
#       same language, legal system, and Orthodox heritage.
#       Players: Baghdatis (1 slot).
#     MCO (Monaco) -> FRA: French is the official language; Monaco is
#       embedded in French institutional and economic orbit.
#     MDA (Moldova) -> ROU: Romanian-speaking, shared Latin heritage and
#       geographic contiguity.
#
#   Tier 2 — same region / closest Hofstede entry (moderate confidence):
#     BLR (Belarus) -> RUS: post-Soviet Slavic; closest Hofstede match.
#     BOL (Bolivia) -> PER: neighbouring Andean country; similar colonial
#       history, indigenous population share, and collectivist orientation.
#       Players: Dellien (6 slots).
#     DOM (Dominican Republic) -> COL: Spanish-speaking Caribbean/Latin
#       American; Colombia is the nearest Hofstede entry in cultural profile.
#       Players: Burgos (1 slot).
#     LBN (Lebanon) -> ARA composite IDV=38: Hofstede's Arab-countries
#       composite is the only available reference; assigned directly.
#       Players: Habib, Hassan (2 slots).
#     TUN (Tunisia) -> MAR: North African, Arabic-speaking neighbour with
#       similar Maghrebi cultural profile.
#     GEO (Georgia) -> TUR: closest geographically available entry;
#       acknowledged as a weaker proxy.
#     KAZ (Kazakhstan) -> RUS: post-Soviet; Hofstede standard proxy for
#       Central Asian FSU states.
#     UKR (Ukraine) -> RUS: post-Soviet Slavic; acknowledged limitation
#       given the current political context.
#     UZB (Uzbekistan) -> RUS: post-Soviet; same rationale as KAZ.
# ---------------------------------------------------------------------------
PROXIES = {
    # Tier 1
    'BIH': ('SRB', 'Bosnia proxied by Serbia: South Slavic kin, shared Yugoslav history'),
    'CYP': ('GRC', 'Cyprus proxied by Greece: Greek-Cypriot majority, same language'),
    'MCO': ('FRA', 'Monaco proxied by France: French official language, institutional integration'),
    'MDA': ('ROU', 'Moldova proxied by Romania: Romanian-speaking, shared Latin heritage'),
    # Tier 2
    'BLR': ('RUS', 'Belarus proxied by Russia: post-Soviet Slavic'),
    'BOL': ('PER', 'Bolivia proxied by Peru: neighbouring Andean country, similar colonial history'),
    'DOM': ('COL', 'Dominican Republic proxied by Colombia: Spanish-speaking Latin American/Caribbean'),
    'LBN': (None,  'Lebanon: Hofstede Arab-countries composite IDV=38 used directly'),
    'TUN': ('MAR', 'Tunisia proxied by Morocco: North African Arabic-speaking neighbour'),
    'GEO': ('TUR', 'Georgia proxied by Turkey: closest geographically available; weaker proxy'),
    'KAZ': ('RUS', 'Kazakhstan proxied by Russia: post-Soviet'),
    'UKR': ('RUS', 'Ukraine proxied by Russia: post-Soviet Slavic'),
    'UZB': ('RUS', 'Uzbekistan proxied by Russia: post-Soviet'),
}

# Direct IDV value for LBN (Hofstede Arab-countries composite)
LBN_IDV_DIRECT = 38

print(f'Direct mappings: {len(CTR_TO_ISO3)}')
print(f'Proxy mappings:  {len(PROXIES)}')

## 3. Build clean IDV table

In [ ]:
raw['idv_num'] = pd.to_numeric(raw['idv'], errors='coerce')

# Keep only rows with a direct ISO3 mapping and a valid IDV score
raw['iso3'] = raw['ctr'].map(CTR_TO_ISO3)
direct = raw[raw['iso3'].notna() & raw['idv_num'].notna()][['iso3', 'country', 'idv_num']].copy()
direct.columns = ['iso3', 'hofstede_country', 'idv']
direct['proxy_tier'] = ''
direct['proxy_note'] = ''

print(f'Direct IDV entries: {len(direct)}')

# Build IDV lookup dict for proxy resolution
idv_lookup = direct.set_index('iso3')['idv'].to_dict()

TIER1 = {'BIH', 'CYP', 'MCO', 'MDA'}

# Add proxy rows
proxy_rows = []
for iso3, (proxy_iso3, note) in PROXIES.items():
    if proxy_iso3 is None:
        # Direct value (LBN: Arab-countries composite)
        idv_val = LBN_IDV_DIRECT
        hcountry = 'Arab countries (composite)'
    else:
        idv_val  = idv_lookup.get(proxy_iso3)
        hcountry = f'proxy:{proxy_iso3}'
    tier = 'Tier 1' if iso3 in TIER1 else 'Tier 2'
    proxy_rows.append({'iso3': iso3, 'hofstede_country': hcountry,
                       'idv': idv_val, 'proxy_tier': tier, 'proxy_note': note})

proxies_df = pd.DataFrame(proxy_rows)
hofstede = pd.concat([direct, proxies_df], ignore_index=True).sort_values('iso3').reset_index(drop=True)

print(f'\nTotal rows (direct + proxy): {len(hofstede)}')
print(f'Missing IDV after proxies:   {hofstede["idv"].isna().sum()}')
hofstede

## 4. Coverage check against tennis dataset

In [ ]:
TENNIS_ISO3 = [
    'ARG','AUS','AUT','BEL','BIH','BLR','BOL','BRA','CAN','CHE','CHL','CHN',
    'COL','CYP','CZE','DEU','DNK','DOM','ECU','ESP','FIN','FRA','GBR','GEO',
    'GRC','HRV','HUN','IDN','IND','ISR','ITA','JAM','JPN','KAZ','KOR','LBN',
    'LTU','MAR','MCO','MDA','MEX','NLD','NOR','NZL','PAK','PER','PHL','POL',
    'PRT','ROU','RUS','SLV','SRB','SVK','SVN','SWE','THA','TUN','TWN','UKR',
    'URY','USA','UZB','VEN','ZAF'
]

covered = hofstede.set_index('iso3')['idv'].to_dict()
no_coverage = [c for c in TENNIS_ISO3 if c not in covered or pd.isna(covered.get(c))]

print(f'Tennis ISO3 codes: {len(TENNIS_ISO3)}')
print(f'Covered by IDV:    {len(TENNIS_ISO3) - len(no_coverage)}')
print(f'Missing coverage:  {no_coverage}')

## 5. Save

In [ ]:
hofstede.to_csv(OUT_PATH, index=False)
print(f'Saved {len(hofstede)} rows to {OUT_PATH}')
hofstede[['iso3', 'idv', 'proxy_note']].to_string(index=False) and None
print(hofstede[['iso3', 'idv', 'proxy_note']].to_string(index=False))